<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_3_transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_3_model_transformers

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [4]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [5]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [7]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [8]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [9]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [10]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [11]:
import json
'''
# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]
'''

'\n# Ruta al archivo guardado\npath = f\'{drive_path}/2_feature_engineering/features_list.json\'\n\nwith open(path, "r") as f:\n    features_dict = json.load(f)\n\n# Extraer las listas\nfeatures_to_30 = features_dict["features_to_30"]\nfeatures_to_60 = features_dict["features_to_60"]\nfeatures_to_90 = features_dict["features_to_90"]\n'

In [12]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### 2.0.1. Función para cargar ventanas

In [13]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### 2.0.2. Función para revisar información de ventanas

In [14]:
def xy_info(fold, X_train, y_train, X_valid, y_valid, X_test, y_test):
    sets = [
        ("Train", X_train, y_train),
        ("Valid", X_valid, y_valid),
        ("Test",  X_test,  y_test),
    ]

    print(f"\nResumen Fold {fold}:")
    for name, X, y in sets:
        if X is None:
            print(f"  {name}: sin datos.")
            continue

        if X.ndim == 2:
            info_dim = f"{X.shape[1]} features (aplanado)"
        else:
            info_dim = f"{X.shape[1]}×{X.shape[2]} (steps × features)"

        print(f"  {name}: {X.shape[0]} ventanas | {info_dim} | {y.shape[0]} targets")

    return X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas

In [15]:
k_folds = [ 1, 2, 3, 4, 5]

In [16]:
#Para verificar el formato de lo guardado.
#for k in k_folds:
#    print(f'Fold {k}:')
#    print('\tTrain:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz').files)
#    print('\tValid:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz').files)
#    print('\tTest:\t',np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz').files)

In [17]:
for k in k_folds:
    print(f'Fold {k}:')
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)
    # Guardar cada uno en variables dinámicas
    globals()[f"X_train_sc_{k}"] = X_train
    globals()[f"y_train_sc_{k}"] = y_train
    globals()[f"X_valid_sc_{k}"] = X_valid
    globals()[f"y_valid_sc_{k}"] = y_valid
    globals()[f"X_test_sc_{k}"]  = X_test
    globals()[f"y_test_sc_{k}"]  = y_test

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
Fold 5:
	X_train_sc_5 e y_train_5 extraídos correctamente
	X_valid_sc_5 e y_valid_5 extraídos correctamente
	X_test_sc_5 e y_test_5 extraídos correctamente


In [71]:
pesos_folds = {}  # acá guardaremos los pesos

for k in k_folds:

    # Guardamos los datasets escalados para este fold
    globals()[f"X_train_sc_{k}"] = X_train
    globals()[f"y_train_sc_{k}"] = y_train
    globals()[f"X_valid_sc_{k}"] = X_valid
    globals()[f"y_valid_sc_{k}"] = y_valid
    globals()[f"X_test_sc_{k}"]  = X_test
    globals()[f"y_test_sc_{k}"]  = y_test

    #===> CAPTURAR LOS PESOS DEL FOLD <===
    w_train, w_valid, w_test = xy_info(
        k,
        globals()[f"X_train_sc_{k}"],
        globals()[f"y_train_sc_{k}"],
        globals()[f"X_valid_sc_{k}"],
        globals()[f"y_valid_sc_{k}"],
        globals()[f"X_test_sc_{k}"],
        globals()[f"y_test_sc_{k}"],
    )

    # Guardar pesos
    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test": w_test
    }



Resumen Fold 1:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 2:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 3:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 4:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 features (aplanado) | 24898 targets
  Test: 27852 ventanas | 1080 features (aplanado) | 27852 targets

Resumen Fold 5:
  Train: 223871 ventanas | 1080 features (aplanado) | 223871 targets
  Valid: 24898 ventanas | 1080 feature

## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [19]:
def load_metrics(data: str):
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [20]:
def metrics_verify(data: str) -> bool:
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{data}.parquet'
    return os.path.exists(data_path)


In [21]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [22]:
transformers_metrics, metrics = load_or_create_metrics("5_3_transformers_metrics")

Las métricas no existen. Se crea el dataset transformers_metrics para almacenar las métricas


In [23]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [60]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/5_transformer_90_model/5_3_model_transformer/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [25]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [26]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

# Entrenamiento de Transformers

## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [27]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [28]:
window_size = 90
#features_base = ['open','high','close','low','volume']

In [29]:
features_90 = ['open',  'high',  'low',  'close',  'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60',  'momentum_5',  'roc_20',  'rev_mom_vol_z_60']
n_features_90 = len (features_90)
print(n_features_90)

12


In [30]:
# Re-shape de tus matrices 2D -> 3D
for k in k_folds:
    #print(f'Fold {k}:')
    globals()[f"Xtr_{k}"]  = reshape_windows(globals()[f"X_train_sc_{k}"], window_size, n_features_90)
    globals()[f"Xva_{k}"]  = reshape_windows(globals()[f"X_valid_sc_{k}"], window_size, n_features_90)
    globals()[f"Xte_{k}"]  = reshape_windows(globals()[f"X_test_sc_{k}"], window_size, n_features_90)
    print(f'Fold {k} re-shape completo')

Fold 1 re-shape completo
Fold 2 re-shape completo
Fold 3 re-shape completo
Fold 4 re-shape completo
Fold 5 re-shape completo


In [31]:
def mostrar_shapes_folds_simple(k_folds):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Escalado (.npz)':<25}{'Reshape (3D)':<25}")
        print("-" * 60)

        for nombre, s1, s2 in [
            ("Train", globals()[f'X_train_sc_{k}'].shape, globals()[f'Xtr_{k}'].shape),
            ("Valid", globals()[f'X_valid_sc_{k}'].shape, globals()[f'Xva_{k}'].shape),
            ("Test",  globals()[f'X_test_sc_{k}'].shape,  globals()[f'Xte_{k}'].shape),
        ]:
            print(f"{nombre:<10}{str(s1):<25}{str(s2):<25}")

mostrar_shapes_folds_simple(k_folds)



Shapes del Fold 1
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 2
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 3
Set       Escalado (.npz)          Reshape (3D)             
------------------------------------------------------------
Train     (223871, 1080)           (223871, 90, 12)         
Valid     (24898, 1080)            (24898, 90, 12)          
Test      (27852, 1080)            (27852, 90, 12)          

Shapes del Fold 4
Set      

### 4.2. Encoder: (backbone + posición + TransformerEncoder)

Mi TimeSeriesEncoder

- Entrada: x con shape (B, T, F)
  - B = batch size
  - T = ventana temporal (p.ej. 90 pasos)
  - F = cantidad de features por minuto

- Salida: z con shape (B, T, D)
  - D = d_model (en tu caso 128)

Es decir: para cada paso temporal devuelve un embedding de dimensión 128

In [32]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

Creo un encoder por cada fold, con la misma arquitectura para todos los folds y pesos distintos (cada encoder_k es un modelo nuevo)

In [33]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Encoder por cada fold
for k in k_folds:
  globals()[f'encoder_{k}'] = TimeSeriesEncoder(
      input_dim=n_features_90,
      d_model=128,
      nhead=8,
      num_layers=2
  ).to(device)

El siguiente código es un testeo rápido para verificar que:
  - Las ventanas del fold están correctamente cargadas.
  - En encoder funciona bien.
  - Las dimensiones de salida son las esperadas.

No está entrenando nada, solo está probando.

In [34]:
def verificar_encoder (fold):
  #fold = 1  # elegir el fold
  print(f"\n=== Fold {fold} ===")
  device = "cuda" if torch.cuda.is_available() else "cpu"

  # ---------- 1) Cargar las ventanas 3D del fold ----------
  # Extrae de memoria Xtr_k -> [N_train, T, F] con T = tamaño de ventana (90), y F cantidad de features (12).
  Xtr = globals()[f"Xtr_{fold}"]
  Xva = globals()[f"Xva_{fold}"]
  Xte = globals()[f"Xte_{fold}"]

  # ---------- 2) Crear tensores chicos (mini-batch) para inspección  ----------
  #Esto hace:
  # Convierte las primeras 64 ventanas a tensores PyTorch y los manda a GPU si está disponible.
  # 64 porque es un batch pequeño para inspeccionar la forma de la salida del encoder.
  xb_tr = torch.tensor(Xtr[:64], dtype=torch.float32).to(device)
  xb_va = torch.tensor(Xva[:64], dtype=torch.float32).to(device)
  xb_te = torch.tensor(Xte[:64], dtype=torch.float32).to(device)

  # ---------- 3) Encoder del fold ----------
  #Selecciona el encoder correspondiente al fold
  enc = globals()[f"encoder_{fold}"]
  # ---------- 4) Lista de sets ----------
  #Se arma una lista con: el mini-batch, el encoder, una etiqueta para imprimir.
  pairs = [
      (xb_tr, enc, f"Fold{fold}-train"),
      (xb_va, enc, f"Fold{fold}-valid"),
      (xb_te, enc, f"Fold{fold}-test"),
  ]

  # ---------- 5) Correr encoder ----------
  #Pasar cada mini‐batch por el encoder
  #Desactiva gradientes (no_grad()) porque no estamos entrenando. Pasa el batch por el encoder e Imprime la dimensión de la salida.

  for xb, encoder, tag in pairs:
      with torch.no_grad():
          z = encoder(xb)
      print(tag, "→", z.shape)


In [35]:
for k in k_folds:
  verificar_encoder(k)


=== Fold 1 ===
Fold1-train → torch.Size([64, 90, 128])
Fold1-valid → torch.Size([64, 90, 128])
Fold1-test → torch.Size([64, 90, 128])

=== Fold 2 ===
Fold2-train → torch.Size([64, 90, 128])
Fold2-valid → torch.Size([64, 90, 128])
Fold2-test → torch.Size([64, 90, 128])

=== Fold 3 ===
Fold3-train → torch.Size([64, 90, 128])
Fold3-valid → torch.Size([64, 90, 128])
Fold3-test → torch.Size([64, 90, 128])

=== Fold 4 ===
Fold4-train → torch.Size([64, 90, 128])
Fold4-valid → torch.Size([64, 90, 128])
Fold4-test → torch.Size([64, 90, 128])

=== Fold 5 ===
Fold5-train → torch.Size([64, 90, 128])
Fold5-valid → torch.Size([64, 90, 128])
Fold5-test → torch.Size([64, 90, 128])


Los resultados significan que:
- batch size = 64
- T = 90 pasos temporales
- d_model = 128 (dimensión del embedding por paso)

## 5. Pooling (Sin cambiar enconder)

Tenemos dos opciones simples (no requieren modificar el encoder):

- `mean`: promedio temporal.
- `last`: último paso temporal.

In [36]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

In [37]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

for fold in k_folds:
    print(f"\n=== Fold {fold} ===")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Cargar ventanas del fold
    Xtr = globals()[f"Xtr_{fold}"]
    Xva = globals()[f"Xva_{fold}"]
    Xte = globals()[f"Xte_{fold}"]

    # 2) Mini-batches
    xb_tr = torch.tensor(Xtr[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold
    enc = globals()[f"encoder_{fold}"]

    # 4) Lista de sets de este fold
    pairs = [
        (xb_tr, enc, f"Fold{fold}-train"),
        (xb_va, enc, f"Fold{fold}-valid"),
        (xb_te, enc, f"Fold{fold}-test"),
    ]

    # 5) Pasar por encoder + pooling
    for xb, encoder, tag in pairs:
        with torch.no_grad():
            z = encoder(xb)      # (64, T, 128)
            p1 = pool_mean(z)    # (64, 128)
            p2 = pool_last(z)    # (64, 128)
        print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')


=== Fold 1 ===
Para Fold1-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 2 ===
Para Fold2-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 3 ===
Para Fold3-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 4 ===
Para Fold4-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-test

Intepretación:

  - Estamos tomando 64 ventanas de cada set (train/valid/test) del fold k, por lo tanto el batch es de tamaño 64.
  - El encoder devuelve secuencias (64, T, 128) y luego:
    - pool_mean(z) → comprime en (64, 128) (promedio temporal).
    - pool_last(z) → comprime en (64, 128) (último paso temporal).
  - Para todos los folds, la dimensión del embedding es 128, como se definió con d_model=128.

Que las shapes sean iguales entre folds es normal: todos usan el mismo d_model y el mismo batch_size.

Hemos verificado que:
  - Que Xtr_k, Xva_k, Xte_k tienen la forma correcta (pueden entrar al encoder).
  - Que los encoder_k están bien definidos y funcionan para todos los folds.
  - Que el TemporalPooling funciona y genera embeddings 2D (batch, 128) listos para una cabeza final (regresión/clasificación).

### 5.0. Verificación de valores entre folds

Veamos el contenido de las primeras filas para compararlas fold a fold, deberían ser distintos (otra distribución temporal, otros días, etc.), aunque la forma se la misma.

El siguiente código es para ver el contenido real (los valores numéricos) del embedding de cada fold, no solo las dimensiones.

#### 5.0.1. Primeros valores del embedding por fold

In [38]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

for fold in k_folds:

    print(f"\n=== Fold {fold} ===")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Cargar ventanas del fold
    Xtr = globals()[f"Xtr_{fold}"]

    # 2) Mini-batch
    xb = torch.tensor(Xtr[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold
    enc = globals()[f"encoder_{fold}"]

    with torch.no_grad():
        z = enc(xb)               # (64, T, 128)
        p_mean = pool_mean(z)     # (64, 128)

    # Mostrar los valores numericos
    print("Primeros 10 valores del embedding del fold:")
    print(p_mean[0, :10].cpu().numpy())   # <-- Esto muestra contenido real


=== Fold 1 ===
Primeros 10 valores del embedding del fold:
[ 0.1300231  -0.02430756 -1.4482251   0.45026085  0.35817644  0.73445106
  0.25763997 -0.18756197  0.15280807 -0.39225686]

=== Fold 2 ===
Primeros 10 valores del embedding del fold:
[-1.3211511   0.31100076  0.15887582  0.9226727  -0.45271075  1.0245696
  1.1837311  -0.8129685   0.647453    0.15080637]

=== Fold 3 ===
Primeros 10 valores del embedding del fold:
[-0.20057075 -0.7681851   0.3369456   0.84241396 -0.9595413  -0.23265201
  0.7093801  -0.2956814  -0.65796214  1.130074  ]

=== Fold 4 ===
Primeros 10 valores del embedding del fold:
[ 1.076518    1.7332028  -1.7383373  -1.2161613   0.45148665 -0.50417936
 -0.57483673  0.3311597   0.957908   -0.2652129 ]

=== Fold 5 ===
Primeros 10 valores del embedding del fold:
[-0.36657333  0.6864595   0.78818125  0.14234483  2.0320244  -1.0883695
  0.98680204 -0.66991377  1.2866722  -0.16638505]


#### 5.0.2. Para observar el contenido de Train, Valid y Test separados

In [39]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

for fold in k_folds:

    print(f"\n\n=== FOLD {fold} ===")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Cargar ventanas
    Xtr = globals()[f"Xtr_{fold}"]
    Xva = globals()[f"Xva_{fold}"]
    Xte = globals()[f"Xte_{fold}"]

    sets = {
        "train": Xtr,
        "valid": Xva,
        "test" : Xte
    }

    enc = globals()[f"encoder_{fold}"]

    for name, X in sets.items():

        xb = torch.tensor(X[:64], dtype=torch.float32).to(device)

        with torch.no_grad():
            z = enc(xb)
            p_mean = pool_mean(z)

        print(f"\n{name.upper()} — primeros 10 valores:")
        print(p_mean[0, :10].cpu().numpy())



=== FOLD 1 ===

TRAIN — primeros 10 valores:
[ 0.11313166 -0.11247052 -1.4405606   0.3987537   0.44621277  0.6375659
  0.14002983 -0.15977503  0.17141491 -0.35938522]

VALID — primeros 10 valores:
[-0.6774683   0.6112542   0.8696674  -1.9823749  -1.3866073  -0.1396039
 -0.16787986  0.4121962  -1.8396422  -0.3803179 ]

TEST — primeros 10 valores:
[-0.9069561   0.86647564  1.0475628  -1.5431738  -1.3963238  -0.2116537
  0.3683588  -0.56713545 -2.1050637  -0.13246906]


=== FOLD 2 ===

TRAIN — primeros 10 valores:
[-1.2789522   0.29532874  0.19221324  0.9066145  -0.45789272  1.0677278
  1.1281109  -0.7792946   0.6258399   0.14542641]

VALID — primeros 10 valores:
[ 0.3876186  -1.6452749   0.5399123  -1.1276597   0.5302036  -1.5669044
  1.3201245   0.08332894 -0.4647384   0.2783261 ]

TEST — primeros 10 valores:
[ 0.40762275 -1.7578902   0.80349284 -1.3205715   0.8908524  -1.6047883
  1.191351    0.46200502 -1.0945928   0.34337422]


=== FOLD 3 ===

TRAIN — primeros 10 valores:
[-0.14017

Los resultados muestran que cada fold produce embeddings distintos en train, valid y test. Eso significa que:
- El encoder funciona correctamente en todos los folds.
- Las ventanas de cada fold son distintas y generan representaciones diferentes.
- No hay colapso del modelo (no devuelve valores repetidos o constantes).
- No hay NaNs ni explosiones, los valores están en rangos normales.
- El pipeline completo fold → encoder → pooling está sano.

En resumen:
Los folds, encoders y embeddings están generándose correctamente y de forma independiente, exactamente como debe ser en un experimento de validación temporal.

## 6. Cabeza de regresión ('Regression Head') - Salida escalar

- Primero se recibe un embedding del encoder → típicamente (B, D)
- Produce un único valor escalar por muestra → (B,)

Ese escalar es:

- el retorno futuro,
- la dirección del precio,
- la magnitud del movimiento,
- o cualquier variable continua que deseemos predecir.

Es el último bloque de la red, el que convierte el embedding en una predicción.

Una cabeza chiquita y estándar:

In [40]:
class RegressionHead(nn.Module):
    """
    Cabeza de regresión para modelos de series temporales.
    Toma un embedding de dimensión D (por ejemplo, 128) y produce
    un único valor escalar por muestra (predicción continua).
    """

    def __init__(self, d_model: int = 128, dropout: float = 0.1):
        super().__init__()

        # Red neuronal totalmente conectada (MLP) en dos capas:
        # 1) Proyección D -> D/2 con activación GELU.
        # 2) Proyección D/2 -> 1 (salida escalar).
        self.net = nn.Sequential(

            # Primera capa lineal: reduce la dimensión del embedding.
            # Entrada: (B, d_model)
            # Salida:  (B, d_model // 2)
            nn.Linear(d_model, d_model // 2),

            # GELU: activación usada en Transformers, suave y estable.
            nn.GELU(),

            # Dropout: regularización para evitar overfitting
            nn.Dropout(dropout),

            # Segunda capa lineal: produce un solo valor por muestra.
            # Entrada: (B, d_model // 2)
            # Salida:  (B, 1)
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de la cabeza de regresión.

        Parámetros
        ----------
        x : Tensor con forma (B, D)
            D es la dimensión del embedding producido por el encoder.

        Retorna
        -------
        Tensor con forma (B,)
            Un valor escalar predicho por cada muestra del batch.
        """

        # La red produce un tensor de forma (B, 1).
        # squeeze(-1) elimina la última dimensión para dejarlo en (B,).
        return self.net(x).squeeze(-1)

Esta cabeza es correcta para nuestra tarea MNQ? si porque:

- El objetivo es es un valor escalar continuo: el retorno futuro a 90min.
- El encoder produce embeddings (64, 128) o (batch, 128).
- Necesitamos convertir esos embeddings en predicciones escalares.

Esta arquitectura es estándar y efectiva en forecasting con Transformers.

### 6.1. Creamos un head por cada fold

In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"

for k in k_folds:
    globals()[f"head_{k}"] = RegressionHead(
        d_model=128,
        dropout=0.1
    ).to(device)

### 6.2. Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”):

Con el sanity check vamos a probar rápidamente que todo el pipeline funciona de punta a punta antes de entrenar.

El pipeline completo es:

`ventanas → encoder → pooling → cabeza de regresión → predicción escalar`


El sanity check verifica lo siguiente:

- Que las ventanas pasen bien por el encoder → (64, 90, 128)
- Que el pooling reduzca correctamente la secuencia → (64, 128)
- Que la cabeza de regresión genere predicciones escalares → (64,)
- Que no haya errores de forma (shape), NaNs ni problemas de device (CPU/GPU).

**Esto NO entrena nada, solo garantiza que la arquitectura está bien conectada.**

El siguiente bloque valida que **todo el pipeline del modelo funcione correctamente** antes de entrenar. Recorre cada fold y verifica que:

1. Se cargan correctamente:
   - `Xtr_k`, `Xva_k`, `Xte_k`
   - `encoder_k`
   - `head_k`

2. Se toma un mini-batch de tamaño **64** de cada set:
   - train  
   - valid  
   - test  

3. Se ejecuta el pipeline completo: `ventanas → encoder → pooling → RegressionHead → predicción escalar`


4. Se comprueba que las *shapes* sean las esperadas:

- **Salida del encoder:**    `z` → `(B, T, 128)`
- **Salida del pooling:**     `h` → `(B, 128)`
- **Salida de la cabeza de regresión:**   `yhat` → `(B,)`

5. El código imprime:
- Si el pipeline es correcto para ese fold,
- Si detecta una forma inesperada,
- Una conclusión final:  
  **“Pipeline COMPLETO OK en el fold X”** o  
  **“Problemas de shapes en el fold X”**.

In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Capa de pooling temporal (usa mean sobre el eje temporal T)
pool = TemporalPooling("mean").to(device)

def sanity_check_pipeline(k_folds, batch_size=64):
    """
    Verifica el pipeline completo encoder → pooling → cabeza de regresión
    para cada fold, usando un mini-batch chico (batch_size).
    Imprime shapes y si el pipeline está OK o no.
    """

    for fold in k_folds:
        print(f"\n=== Sanity check FOLD {fold} ===")

        # 1) Recuperar ventanas 3D del fold desde las variables globales
        try:
            Xtr = globals()[f"Xtr_{fold}"]
            Xva = globals()[f"Xva_{fold}"]
            Xte = globals()[f"Xte_{fold}"]

            enc  = globals()[f"encoder_{fold}"]  # encoder del fold
            head = globals()[f"head_{fold}"]     # cabeza de regresión del fold
        except KeyError as e:
            print(f" Falta alguna variable para el fold {fold}: {e}")
            continue

        # Diccionario de sets para recorrer train / valid / test
        sets = {
            "train": Xtr,
            "valid": Xva,
            "test":  Xte
        }

        fold_ok = True  # asumimos OK y marcamos False si algo falla

        for nombre_set, X in sets.items():
            if X is None or len(X) == 0:
                print(f" {nombre_set}: sin datos, se omite.")
                continue

            # 2) Mini-batch chico para inspección
            xb = torch.tensor(X[:batch_size], dtype=torch.float32).to(device)

            with torch.no_grad():
                # Paso 1: encoder → secuencia embebida
                z = enc(xb)           # esperado: (B, T, D)

                # Paso 2: pooling → un embedding por ventana
                h = pool(z)           # esperado: (B, D)

                # Paso 3: cabeza de regresión → escalar por muestra
                yhat = head(h)        # esperado: (B,)

            # Comprobaciones básicas de shapes
            ok_shapes = (
                z.ndim == 3 and
                h.ndim == 2 and
                yhat.ndim == 1 and
                xb.shape[0] == h.shape[0] == yhat.shape[0]
            )

            if ok_shapes:
                print(
                    f"  {nombre_set}: "
                    f"z{tuple(z.shape)} → h{tuple(h.shape)} → yhat{tuple(yhat.shape)}"
                )
            else:
                print(
                    f" {nombre_set}: shapes inesperadas: "
                    f"z{tuple(z.shape)}, h{tuple(h.shape)}, yhat{tuple(yhat.shape)}"
                )
                fold_ok = False

        if fold_ok:
            print(f"Pipeline COMPLETO OK en el fold {fold}.")
        else:
            print(f"Problemas de shapes en el fold {fold}.")


# Llamada al sanity check para todos los folds
sanity_check_pipeline(k_folds)



=== Sanity check FOLD 1 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
Pipeline COMPLETO OK en el fold 1.

=== Sanity check FOLD 2 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
Pipeline COMPLETO OK en el fold 2.

=== Sanity check FOLD 3 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
Pipeline COMPLETO OK en el fold 3.

=== Sanity check FOLD 4 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
Pipeline COMPLETO OK en el fold 4.

=== Sanity check FOLD 5 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 12

Antes de entrenar, es fundamental asegurarnos de que:

- Las ventanas están bien formateadas (3D correctas).  
- El encoder procesa correctamente la secuencia.  
- El pooling reduce correctamente la dimensión temporal.  
- La cabeza de regresión produce un escalar por muestra.  
- Todo funciona en CPU o GPU sin errores.

Este paso nos garantiza que el pipeline entero está sano y listo para el entrenamiento real.

### 6.3. Sanity check de pérdida (MSE).

Lo que buscamos es verificar que los targets reales (y) y las predicciones del modelo (ŷ) tengan formas compatibles, estén en el mismo device, y permitan calcular correctamente la pérdida MSE.

En otras palabras comprueba que el pipeline produce predicciones escalares válidas y comparables con los targets.

In [43]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

pool = TemporalPooling("mean").to(device)

for fold in k_folds:
    print(f"\n=== Generando yhat para Fold {fold} ===")

    # 1) Datos del fold
    Xtr = globals()[f"Xtr_{fold}"]
    Xva = globals()[f"Xva_{fold}"]
    Xte = globals()[f"Xte_{fold}"]

    enc  = globals()[f"encoder_{fold}"].to(device)
    head = globals()[f"head_{fold}"].to(device)

    # 2) Mini‐batches (solo las primeras 64 muestras)
    xb_tr = torch.tensor(Xtr[:batch_size], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva[:batch_size], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte[:batch_size], dtype=torch.float32).to(device)

    with torch.no_grad():
        # ---- TRAIN ----
        z_tr   = enc(xb_tr)          # (B, T, D)
        h_tr   = pool(z_tr)          # (B, D)
        yhat_tr = head(h_tr)         # (B,)
        globals()[f"yhat_train_{fold}"] = yhat_tr.cpu().numpy()

        # ---- VALID ----
        z_va   = enc(xb_va)
        h_va   = pool(z_va)
        yhat_va = head(h_va)
        globals()[f"yhat_valid_{fold}"] = yhat_va.cpu().numpy()

        # ---- TEST ----
        z_te   = enc(xb_te)
        h_te   = pool(z_te)
        yhat_te = head(h_te)
        globals()[f"yhat_test_{fold}"] = yhat_te.cpu().numpy()

    print(f"  yhat_train_{fold}.shape =", globals()[f'yhat_train_{fold}'].shape)
    print(f"  yhat_valid_{fold}.shape =", globals()[f'yhat_valid_{fold}'].shape)
    print(f"  yhat_test_{fold}.shape  =", globals()[f'yhat_test_{fold}'].shape)



=== Generando yhat para Fold 1 ===
  yhat_train_1.shape = (64,)
  yhat_valid_1.shape = (64,)
  yhat_test_1.shape  = (64,)

=== Generando yhat para Fold 2 ===
  yhat_train_2.shape = (64,)
  yhat_valid_2.shape = (64,)
  yhat_test_2.shape  = (64,)

=== Generando yhat para Fold 3 ===
  yhat_train_3.shape = (64,)
  yhat_valid_3.shape = (64,)
  yhat_test_3.shape  = (64,)

=== Generando yhat para Fold 4 ===
  yhat_train_4.shape = (64,)
  yhat_valid_4.shape = (64,)
  yhat_test_4.shape  = (64,)

=== Generando yhat para Fold 5 ===
  yhat_train_5.shape = (64,)
  yhat_valid_5.shape = (64,)
  yhat_test_5.shape  = (64,)


In [44]:
def sanity_check_mse_folds(k_folds, batch_size=64):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    for fold in k_folds:
        print(f"\n=== Sanity check MSE — Fold {fold} ===")

        # 1) Obtener targets del fold
        ytr = globals()[f"y_train_sc_{fold}"]
        yva = globals()[f"y_valid_sc_{fold}"]
        yte = globals()[f"y_test_sc_{fold}"]

        # 2) Obtener predicciones del sanity check anterior
        #    (encoder_k → pool → head_k)
        yhat_tr = globals()[f"yhat_train_{fold}"]
        yhat_va = globals()[f"yhat_valid_{fold}"]
        yhat_te = globals()[f"yhat_test_{fold}"]

        sets = [
            ("train", ytr[:batch_size], yhat_tr[:batch_size]),
            ("valid", yva[:batch_size], yhat_va[:batch_size]),
            ("test",  yte[:batch_size], yhat_te[:batch_size]),
        ]

        for name, y_true, y_pred in sets:
            # Convertir a tensores
            yb = torch.tensor(y_true, dtype=torch.float32).to(device)
            yh = torch.tensor(y_pred, dtype=torch.float32).to(device)

            # Asegurar shapes 1D
            yb = yb.view(-1)
            yh = yh.view(-1)

            # Calcular pérdida MSE
            loss = torch.nn.functional.mse_loss(yh, yb)

            print(f"  {name:<6} — yhat:{tuple(yh.shape)}  MSE={float(loss):.6f}")

In [45]:
sanity_check_mse_folds(k_folds)


=== Sanity check MSE — Fold 1 ===
  train  — yhat:(64,)  MSE=0.012676
  valid  — yhat:(64,)  MSE=0.018156
  test   — yhat:(64,)  MSE=0.020384

=== Sanity check MSE — Fold 2 ===
  train  — yhat:(64,)  MSE=0.005470
  valid  — yhat:(64,)  MSE=0.014035
  test   — yhat:(64,)  MSE=0.020131

=== Sanity check MSE — Fold 3 ===
  train  — yhat:(64,)  MSE=0.063060
  valid  — yhat:(64,)  MSE=0.710858
  test   — yhat:(64,)  MSE=0.897282

=== Sanity check MSE — Fold 4 ===
  train  — yhat:(64,)  MSE=0.075854
  valid  — yhat:(64,)  MSE=0.013538
  test   — yhat:(64,)  MSE=0.009378

=== Sanity check MSE — Fold 5 ===
  train  — yhat:(64,)  MSE=0.052406
  valid  — yhat:(64,)  MSE=0.096201
  test   — yhat:(64,)  MSE=0.012626


A partir de los valores obtenidos de MSE para cada fold, podemos establecer las siguientes conclusiones:

**1. El pipeline está funcionando correctamente en todos los folds**

- En todos los casos, las predicciones `yhat` presentan la forma esperada `(64,)`.
- No se registraron errores de dimensiones, tipos de datos o conflictos entre CPU/GPU.
- Esto confirma que el flujo completo encoder → pooling → RegressionHead está operando sin inconsistencias técnicas.

**2. Los valores de MSE son coherentes con un modelo no entrenado**

- Los pesos del encoder y la cabeza de regresión no han sido entrenados aún, por lo que las predicciones son aleatorias.
- En consecuencia:
  - Es esperable que el MSE varíe ampliamente entre folds.
  - No se busca obtener un valor bajo sino simplemente verificar que el cálculo sea posible.
- Ejemplos observados:
  - Fold 1: MSE entre 0.004 y 0.009.
  - Folds 3 y 5: MSE más elevados en validación y prueba, lo cual es normal dada la ausencia de entrenamiento.

**3. El modelo está listo para avanzar al entrenamiento real**

- El pipeline completo ha sido verificado tanto en términos de shapes como de cálculo de pérdida.
- Ya se validó exitosamente:
  - encoder → pooling → head → yhat
  - yhat en comparación con los targets reales mediante MSE.
- El siguiente paso es implementar el bucle de entrenamiento por fold, incluyendo:
  - función de pérdida,
  - optimizador,
  - scheduling de aprendizaje si se requiere,
  - métricas de evaluación (RMSE, MAE, SMAPE, Directional Accuracy).

A partir de este resultado, se confirma que el modelo puede entrenarse sin problemas estructurales.

## 7. Preparación para Entrenamiento

### 7.1. Dataset + DataLoader (reshape dentro)

El siguiente apartado prepara todo lo necesario para entrenar un modelo en PyTorch usando nuestras ventanas:

1. Escala los valores objetivo (y) usando StandardScaler.
    - Esto ayuda a estabilizar el entrenamiento.
    - El scaler se ajusta solo con y_train (buena práctica).

2. Convierte tus ventanas X (aplanadas en 2D) a tensores 3D (B, T, F)
donde:
    - B = batch size
    - T = tamaño de la ventana temporal (90 minutos)
    - F = número de features

3. Construye un Dataset personalizado (WindowDataset)
    - Guarda X y y en formato listo para PyTorch.
    - Aplica el escalador únicamente a y.

4. Crea dataloaders para entrenamiento y validación
    - `dl_tr`: con shuffle=True
    - `dl_va`: sin shuffle, para evaluación estable
    - Ambos con pin_memory=True (optimiza transferencias CPU→GPU)

Este bloque no entrena nada todavía, pero prepara correctamente los datos para alimentar el modelo fold por fold.

In [46]:
import os, joblib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

'''1) Scaler de y (fit solo con y_train)
 Propósito: Normalizar y para facilitar el entrenamiento y evitar escalas muy pequeñas o muy grandes.
'''
def get_y_scaler(y_train: np.ndarray, path: str = None):
    # Crea un StandardScaler y lo ajusta solo con y_train.
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))   # y debe ser columna

    # Si se pasa un path, guarda el scaler en disco.
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(scaler, path)

    return scaler
'''
2) Dataset que aplica el y_scaler
Propósito: PyTorch necesita un Dataset para entregar lotes de entrenamiento.
Aquí se reconstruyen las ventanas (T,F) y se devuelven como tensores.
'''
class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F, y_scaler: StandardScaler):
        # Verifica que X_flat tenga la forma correcta: (N, T*F)
        assert X_flat.shape[1] == T * F, f"Inconsistente: {X_flat.shape[1]} != {T}*{F}"

        # Convierte ventana 2D a 3D: (N, T*F) → (N, T, F)
        X = X_flat.reshape(-1, T, F).astype(np.float32)

        # Si usamos scaler, transformamos y y lo convertimos a float32
        if y_scaler is not None:
            y = y_scaler.transform(y.reshape(-1, 1)).ravel()

        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self):
        # Cantidad total de muestras
        return len(self.y)

    def __getitem__(self, i):
        # Devuelve la i-ésima ventana y su target como tensores PyTorch
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.float32)

'''
3) Loaders genéricos (cualquier horizonte)
Propósito: Generar los iteradores que el modelo usará durante el entrenamiento:
      - dl_tr: batches mezclados
      - dl_va: batches ordenados (evaluación estable)
'''

def make_loaders(Xtr, ytr, Xva, yva, T, F, y_scaler, bs=256, num_workers=2):
    # Crea datasets de entrenamiento y validación
    ds_tr = WindowDataset(Xtr, ytr, T, F, y_scaler=y_scaler)
    ds_va = WindowDataset(Xva, yva, T, F, y_scaler=y_scaler)

    # DataLoader de entrenamiento (mezcla los datos)
    dl_tr = DataLoader(
        ds_tr,
        batch_size=bs,
        shuffle=True,
        pin_memory=True,
        num_workers=num_workers
    )

    # DataLoader de validación (sin shuffle)
    dl_va = DataLoader(
        ds_va,
        batch_size=bs,
        shuffle=False,
        pin_memory=True,
        num_workers=num_workers
    )

    return dl_tr, dl_va


El bloque anterior construye el pipeline que convierte tus dataframes en: `Ventanas 3D → Dataset PyTorch → DataLoader → Entrenamiento`

Transforma:
 - X a (B, T, F)
 - y a valores escalados

### 7.2. Modelo compacto por fold (encoder + pooling mean + head)

Un modelo compacto por fold: `modelo_k = encoder_k + pooling + head_k`

Un modelo compacto por fold combina las tres partes del pipeline (encoder → pooling → head) en un único `nn.Module`.  

Se decidió utilizar un modelo compacto por las siguientes razones:

1. Permite que **cada fold tenga un modelo completamente independiente**, evitando fuga de información entre folds.  
2. Simplifica el loop de entrenamiento: en lugar de ejecutar manualmente `encoder → pool → head`, el modelo produce directamente la predicción `ŷ = model(x)`.  
3. Facilita el uso de optimizadores, carga/guardado de pesos y evaluación, ya que todos los parámetros entrenables quedan dentro de un único módulo por fold.  
4. Mantiene una estructura clara: el “modelo” es la combinación natural de encoder, reducción temporal y cabeza de regresión.

Con esto, el punto 7.3 (loop de entrenamiento) puede trabajar con un único módulo (`model_k`) por fold, lo cual hace el código más limpio y menos propenso a errores.


In [47]:
for fold in k_folds:
    encoder = globals()[f"encoder_{fold}"]
    head    = globals()[f"head_{fold}"]

    # pooling es compartido
    model = nn.Sequential(
        encoder,   # (B,T,F) -> (B,T,128)
        pool,      # reduce temporalmente -> (B,128)
        head       # (B,128) -> (B,)
    )

    globals()[f"model_{fold}"] = model

### 7.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)

El siguiente bloque implementa el loop de entrenamiento del modelo por fold.
Entrena un encoder + pooling + cabeza de regresión usando MSE como función de pérdida, optimizador AdamW, soporte opcional para AMP (mixed precision), clipping de gradiente y un esquema simple de early stopping basado en la pérdida de validación. El objetivo es obtener un modelo estable y con buena generalización, ajustando solo los parámetros del encoder y de la cabeza,mientras que el pooling permanece fijo.


In [48]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_model(model: nn.Module,
                dl_tr: DataLoader,
                dl_va: DataLoader,
                device: str = "cuda" if torch.cuda.is_available() else "cpu",
                lr: float = 3e-4,
                weight_decay: float = 1e-4,
                max_epochs: int = 50,
                patience: int = 8,
                grad_clip: float = 1.0,
                use_amp: bool = True):
    """
    Entrena un modelo compacto (encoder + pooling + cabeza de regresión)
    usando MSE como función de pérdida, AdamW como optimizador y un esquema
    simple de early stopping basado en la pérdida de validación.

    Parámetros
    ----------
    model : nn.Module
        Modelo completo (por ejemplo: nn.Sequential(encoder, pool, head)).
    dl_tr : DataLoader
        DataLoader de entrenamiento.
    dl_va : DataLoader
        DataLoader de validación.
    device : str
        "cuda" si hay GPU disponible, de lo contrario "cpu".
    lr : float
        Learning rate del optimizador AdamW.
    weight_decay : float
        Término de regularización L2 (weight decay) de AdamW.
    max_epochs : int
        Máximo número de épocas de entrenamiento.
    patience : int
        Número de épocas sin mejora en validación antes de activar early stopping.
    grad_clip : float
        Valor máximo de norma de gradiente para aplicar gradient clipping.
        Si es None, no se aplica clipping.
    use_amp : bool
        Si es True y hay GPU, activa mixed precision (AMP) para acelerar el entrenamiento.

    Retorna
    -------
    model : nn.Module
        Modelo con los mejores pesos encontrados (según pérdida de validación).
    """

    # Enviar todo el modelo al dispositivo (GPU/CPU)
    model = model.to(device)

    # Optimizador AdamW (recomendado para arquitecturas tipo Transformer)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # GradScaler para entrenamiento en mixed precision (solo en GPU)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))

    # Variables para seguimiento del mejor modelo (early stopping)
    best_val = float("inf")   # mejor pérdida de validación observada
    best_state = None         # state_dict del mejor modelo
    noimp = 0                 # épocas consecutivas sin mejora

    # ==========================================================
    #                      LOOP DE ÉPOCAS
    # ==========================================================
    for epoch in range(1, max_epochs + 1):

        # ----------------------- ENTRENAMIENTO -----------------------
        model.train()
        tr_loss = 0.0

        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)

            # Reset del gradiente
            opt.zero_grad(set_to_none=True)

            # Forward con AMP opcional
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                # El modelo compacto incluye: encoder → pool → head
                yhat = model(xb).view(-1)  # salida (B,)
                loss = nn.functional.mse_loss(yhat, yb)

            # Backpropagation con GradScaler
            scaler.scale(loss).backward()
            scaler.unscale_(opt)  # necesario antes del clipping

            # Clipping de gradiente para evitar explosiones
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Paso de optimización
            scaler.step(opt)
            scaler.update()

            # Acumulación de la pérdida ponderada por el tamaño del batch
            tr_loss += loss.item() * xb.size(0)

        # Promedio de pérdida de entrenamiento por muestra
        tr_loss /= len(dl_tr.dataset)

        # ----------------------- VALIDACIÓN -----------------------
        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)

                yhat = model(xb).view(-1)
                va_loss += nn.functional.mse_loss(yhat, yb).item() * xb.size(0)

        # Promedio de pérdida de validación por muestra
        va_loss /= len(dl_va.dataset)

        # Log de la época
        print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

        # ----------------------- EARLY STOPPING -----------------------
        if va_loss < best_val - 1e-9:
            # Mejora en validación: se guarda el mejor modelo hasta ahora
            best_val = va_loss
            noimp = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            # No hubo mejora: se incrementa el contador
            noimp += 1
            if noimp >= patience:
                print("Early stopping por falta de mejora en validación.")
                break

    # Restaurar los mejores pesos encontrados
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [49]:
## Para ejecutar el código

'''
for fold in k_folds:
    model_k = globals()[f"model_{fold}"]
    dl_tr_k, dl_va_k = ...  # loaders del fold k
    print(f"\n=== Entrenando modelo del Fold {fold} ===")
    model_k = train_model(model_k, dl_tr_k, dl_va_k)
    globals()[f"model_{fold}"] = model_k
'''

'\nfor fold in k_folds:\n    model_k = globals()[f"model_{fold}"]\n    dl_tr_k, dl_va_k = ...  # loaders del fold k\n    print(f"\n=== Entrenando modelo del Fold {fold} ===")\n    model_k = train_model(model_k, dl_tr_k, dl_va_k)\n    globals()[f"model_{fold}"] = model_k\n'

### 7.4. Inferencia (Predicción).

La siguiente función realiza inferencia (predicción) en un conjunto completo de ventanas sin calcular gradientes, usando el pipeline: `encoder → pooling → head → predicción escalar`

Sirve para obtener todas las predicciones de train, valid o test después de entrenar el modelo por fold.

En detalle:

1. Convierte X_flat (que viene en formato (N, T*F)) a ventanas 3D (N, T, F)
2. Pasa por el modelo en batches grandes (4096 por defecto) para acelerar la inferencia
3. Obtiene las predicciones ŷ
4. Si las predicciones están escaladas, aplica inverse_transform del scaler de y
5. Devuelve un array 1D con las predicciones reales

Es decir: **Esta función transforma un dataset completo en sus predicciones finales del modelo.**

Se usa después de entrenar, tipicamente para:
  - Evaluar rendimiento
  - Graficar pred vs real
  - Guardar resultados
  - Calcular RMSE, MAE, SMAPE, DA, etc.

In [50]:
@torch.no_grad()
def predict_set(enc, pool, head,
                X_flat: np.ndarray,
                T: int, F: int,
                device: str,
                batch_size: int = 4096,
                y_scaler: StandardScaler = None) -> np.ndarray:
    """
    Calcula predicciones en un conjunto completo de ventanas X_flat,
    usando el modelo encoder + pooling + head.

    X_flat debe tener forma (N, T*F).
    Devuelve un vector 1D con las predicciones finales.
    """

    # Número total de ventanas
    N = X_flat.shape[0]

    # Reconstruye X de 2D (N, T*F) a 3D (N, T, F)
    X = X_flat.reshape(N, T, F).astype(np.float32)

    preds = []  # acumulador de predicciones por batch

    # Ponemos encoder y head en modo evaluación (pool no tiene parámetros)
    enc.eval()
    head.eval()

    # Recorremos el dataset en batches grandes
    for i in range(0, N, batch_size):
        # Cargar batch actual en GPU
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)

        # Forward completo: encoder → pooling → head
        z  = enc(xb)          # (B, T, D)
        h  = pool(z)          # (B, D)
        yb = head(h).cpu().numpy()   # pasar a numpy para acumular

        preds.append(yb)

    # Concatenamos todos los batches en un solo array
    y_pred_scaled = np.concatenate(preds, axis=0).reshape(-1, 1)

    # Si se usó scaler, revertimos la escala a valores originales
    if y_scaler is not None:
        y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
    else:
        y_pred = y_pred_scaled.ravel()

    return y_pred


En resumen:
- Esta función realiza predicción vectorizada, sin gradientes.
- Usa el pipeline completo: encoder → pooling → head.
- Procesa el dataset en batches grandes (eficiente).
- Reconstruye ventanas desde 2D → 3D.
- Aplica inverse_transform del scaler de y si corresponde.
- Devuelve un vector plano con todas las predicciones del modelo.

## 8. Entrenamiento

In [51]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### 8.3. Entrenamiento H=90 (train/valid/test)

In [53]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

# Tamaño de ventana y cantidad de features (ajustar si cambia)
T = 90              # longitud de la ventana temporal
F = 12              # número de features por paso (ejemplo: 12)
bs = 256            # batch size para entrenamiento

# Pooling compartido (sin parámetros entrenables)
pool = TemporalPooling("mean").to(device)

for fold in k_folds:
    # 1) Definir una clave de modelo por fold (para registro de métricas)
    model_key = f"transformer_fold_{fold}"

    # Si ya existe en la tabla de métricas, omitimos el entrenamiento
    if ("transformers_metrics" in globals()
        and transformers_metrics is not None
        and model_key in transformers_metrics.index):
        print(f"Omitimos este entrenamiento: {model_key} ya existe en transformers_metrics")
        continue

    print(f"\n=== Entrenando modelo: {model_key} ===")

    # ------------------------------------------------------------
    # 2) Recuperar X e y del fold (ajustar nombres según tu notebook)
    #    Aquí asumo que:
    #    - X_train_sc_{fold}, X_valid_sc_{fold}, X_test_sc_{fold} son 2D (N, T*F)
    #    - y_train_{fold}, y_valid_{fold}, y_test_{fold} son 1D (N,)
    # ------------------------------------------------------------
    Xtr_flat = globals()[f"X_train_sc_{fold}"]
    Xva_flat = globals()[f"X_valid_sc_{fold}"]
    Xte_flat = globals()[f"X_test_sc_{fold}"]

    ytr = globals()[f"y_train_sc_{fold}"]
    yva = globals()[f"y_valid_sc_{fold}"]
    yte = globals()[f"y_test_sc_{fold}"]

    # 3) Scaler de y solo con los datos de entrenamiento del fold
    scaler_y_fold = get_y_scaler(ytr)

    # 4) Crear DataLoaders para entrenamiento y validación del fold
    dl_tr_fold, dl_va_fold = make_loaders(
        Xtr_flat, ytr,
        Xva_flat, yva,
        T=T, F=F,
        y_scaler=scaler_y_fold,
        bs=bs
    )

    # ------------------------------------------------------------
    # 5) Construir el modelo compacto del fold:
    #    encoder_{fold} + pool (compartido) + head_{fold}
    # ------------------------------------------------------------
    encoder_fold = globals()[f"encoder_{fold}"]
    head_fold    = globals()[f"head_{fold}"]

    # Modelo compacto: (B, T, F) → (B,)
    model_fold = nn.Sequential(
        encoder_fold,  # (B, T, F) → (B, T, D)
        pool,          # (B, T, D) → (B, D)
        head_fold      # (B, D) → (B,)
    )

    # 6) Entrenar el modelo del fold con el loop general
    model_fold = train_model(
        model_fold,
        dl_tr_fold,
        dl_va_fold,
        device=device,
        lr=3e-4,
        weight_decay=1e-4,
        max_epochs=50,
        patience=8,
        grad_clip=1.0,
        use_amp=True
    )

    # Nota: encoder_fold y head_fold se actualizan "in-place" al entrenar model_fold,
    # por lo que ahora contienen los pesos entrenados del fold.

    # ------------------------------------------------------------
    # 7) Predicciones finales en train / valid / test para este fold
    #    (usamos predict_set con encoder + pool + head)
    # ------------------------------------------------------------
    ytr_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xtr_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yva_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xva_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yte_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xte_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

  # ------------------------------------------------------------
    # 8) Cálculo de métricas por fold (train / valid / test)
    #    Usamos evaluate_model, pasando y_pred explícitamente.
    # ------------------------------------------------------------
    metrics_tr = evaluate_model(
        model=None,
        X=None,
        y_true=ytr,
        y_pred=ytr_pred
    )

    metrics_va = evaluate_model(
        model=None,
        X=None,
        y_true=yva,
        y_pred=yva_pred
    )

    metrics_te = evaluate_model(
        model=None,
        X=None,
        y_true=yte,
        y_pred=yte_pred
    )

    # 9) (Opcional) Guardar métricas en un DataFrame global
    if "transformers_metrics" in globals() and transformers_metrics is not None:
        for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
            for k, v in m.items():
                transformers_metrics.loc[model_key, f"{split}_{k}"] = v

    # 10) Guardar el modelo entrenado del fold si querés reutilizarlo
    globals()[f"model_{fold}"]  = model_fold
    globals()[f"scaler_y_{fold}"] = scaler_y_fold
    globals()[f"ytr_pred_{fold}"] = ytr_pred
    globals()[f"yva_pred_{fold}"] = yva_pred
    globals()[f"yte_pred_{fold}"] = yte_pred



=== Entrenando modelo: transformer_fold_1 ===
Epoch 001  train=4.352353e-01  valid=1.936649e-01
Epoch 002  train=2.837831e-01  valid=1.689898e-01
Epoch 003  train=2.320224e-01  valid=1.777509e-01
Epoch 004  train=1.956413e-01  valid=1.925408e-01
Epoch 005  train=1.697319e-01  valid=1.582786e-01
Epoch 006  train=1.492941e-01  valid=1.873675e-01
Epoch 007  train=1.328784e-01  valid=1.988040e-01
Epoch 008  train=1.195310e-01  valid=1.867797e-01
Epoch 009  train=1.093911e-01  valid=1.968757e-01
Epoch 010  train=1.003284e-01  valid=1.863297e-01
Epoch 011  train=9.132015e-02  valid=2.023744e-01
Epoch 012  train=8.411923e-02  valid=2.056228e-01
Epoch 013  train=7.815703e-02  valid=2.022213e-01
Early stopping por falta de mejora en validación.

=== Entrenando modelo: transformer_fold_2 ===
Epoch 001  train=4.362421e-01  valid=1.790816e-01
Epoch 002  train=2.851217e-01  valid=1.809078e-01
Epoch 003  train=2.316578e-01  valid=1.973960e-01
Epoch 004  train=1.961569e-01  valid=1.739575e-01
Epoch 

## 9. Métricas

In [80]:
cols_base = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]
transformers_folds_metrics = transformers_metrics.drop(columns=cols_base)
transformers_folds_metrics



,train_RMSE,train_MAE,train_R2,train_SMAPE,train_DirAcc,valid_RMSE,valid_MAE,valid_R2,valid_SMAPE,valid_DirAcc,test_RMSE,test_MAE,test_R2,test_SMAPE,test_DirAcc
transformer_fold_1,0.001869,0.001400,0.847879,78.139954,0.842275,0.001907,0.001395,0.683618,89.568233,0.812917,0.004352,0.002301,0.574116,92.952794,0.812868
transformer_fold_2,0.001662,0.001252,0.879743,72.651872,0.855980,0.001963,0.001416,0.664668,87.048460,0.821753,0.004317,0.002387,0.580819,95.762888,0.793085
transformer_fold_3,0.001658,0.001250,0.880355,71.799080,0.855086,0.001911,0.001399,0.682185,86.963528,0.816933,0.004514,0.002383,0.541749,93.526665,0.803892
transformer_fold_4,0.001789,0.001340,0.860642,73.357976,0.851200,0.001925,0.001395,0.677382,86.728896,0.815367,0.004506,0.002435,0.543366,96.219742,0.791900
transformer_fold_5,0.002117,0.001552,0.804784,78.221071,0.836558,0.001851,0.001362,0.701836,85.801070,0.820267,0.004110,0.002210,0.620103,89.118910,0.816279


In [65]:
save_metrics(transformers_folds_metrics, "5_3_transformers_folds_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_transformers_folds_metrics.parquet


In [81]:
# Crear un nuevo DataFrame con los mismos índices y solo esas columnas
transformers_metrics = pd.DataFrame(index=transformers_metrics.index, columns=cols_base)
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
transformer_fold_1,NaN,NaN,NaN,NaN,NaN
transformer_fold_2,NaN,NaN,NaN,NaN,NaN
transformer_fold_3,NaN,NaN,NaN,NaN,NaN
transformer_fold_4,NaN,NaN,NaN,NaN,NaN
transformer_fold_5,NaN,NaN,NaN,NaN,NaN


### 9.1. Análisis de primeros resultados

1. Consistencia del desempeño entre folds

    - El modelo presenta un comportamiento estable en todas las particiones temporales.
    - Las métricas de entrenamiento y validación muestran poca variación.
    - Los valores de RMSE en entrenamiento se encuentran entre 0.00166 y 0.00212, mientras que el RMSE de validación se mantiene entre 0.00185 y 0.00196.
    - Esta estabilidad indica que el modelo captura patrones generales del mercado sin depender de particularidades de cada fold.

2. Diferencias claras entre Train, Valid y Test

    - En todos los folds se observa una caída en el rendimiento cuando se evalúa sobre el conjunto de test.
    - El RMSE y el MAE aumentan de forma consistente en test, mientras que el R² disminuye.
    - La Directional Accuracy se mantiene cercana al 0.80, aunque también muestra una leve reducción respecto a Train y Valid.
    - Este comportamiento es coherente con:
      - La no estacionariedad de los datos financieros intradía.
      - Posibles cambios de régimen en los días reservados para test.
      - Patrones aprendidos por el modelo que no necesariamente se repiten en el futuro.

3. Relación entre Train y Test (RMSE aproximadamente 2.3 veces mayor)

    - Por ejemplo, en el fold 1:
      - Train RMSE: 0.001869
      - Test RMSE: 0.004352

    - El incremento del error indica un nivel moderado de sobreajuste.
    - Aun así, el modelo mantiene capacidad predictiva en términos direccionales (Direction Accuracy alrededor de 0.80).

4. Comportamiento del R²

    - Los valores promedio aproximados de R² son:
      - Entrenamiento: alrededor de 0.85
      - Validación: entre 0.67 y 0.70
      - Test: entre 0.54 y 0.58
    - Aunque el R² disminuye en test, estos valores son razonables considerando el alto nivel de ruido y variabilidad de las series intradía.
    - Un R² en el orden del 50 % resulta aceptable en este tipo de problemas.

5. SMAPE estable entre folds

    - Los valores observados de SMAPE son:
      - Train: entre 72 y 78
      - Valid: entre 85 y 89
      - Test: entre 89 y 96
    - La diferencia entre validación y test es relativamente pequeña, lo que indica que la dificultad del horizonte de predicción se mantiene consistente.

6. Dirección de movimiento (DirAcc) elevada en Test

    - La precisión direccional (Direction Accuracy) se mantiene entre 0.79 y 0.82.
    - Este desempeño es especialmente relevante para aplicaciones donde la predicción del signo del retorno resulta más importante que el valor exacto.

**Conclusión**

- El modelo muestra un desempeño sólido y consistente en los conjuntos de entrenamiento y validación.
- En el conjunto de test se observa un descenso esperado debido a la naturaleza no estacionaria del mercado, aunque la performance sigue siendo estable.
- La consistencia entre folds sugiere que el modelo captura relaciones reales presentes en los datos intradía.
- La reducción del R² y el aumento del error en test reflejan un sobreajuste moderado o la presencia de cambios de régimen en el mercado.
- Una Direction Accuracy cercana al 80 % posiciona al modelo Transformer como un candidato competitivo para tareas de predicción direccional de retornos intradía.

### 9.2. Promedio ponderado por cantidad de muestras de cada conjunto (train, valid, test).

In [82]:
import pandas as pd

# =====================================================
# FUNCIÓN PARA PROMEDIO PONDERADO DE MÉTRICAS POR FOLD
# =====================================================
def weighted_avg_metrics_from_df(df, w_train, w_valid, w_test):
    """
    Calcula el promedio ponderado de métricas a partir de un DataFrame
    con columnas del tipo train_RMSE, valid_RMSE, test_RMSE, etc.

    df : DataFrame con un fold por fila
    w_train, w_valid, w_test : pesos (cantidad de muestras por conjunto)
    """

    # Métricas base
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    resultados = {}

    for m in metricas:
        col_train = f"train_{m}"
        col_valid = f"valid_{m}"
        col_test  = f"test_{m}"

        # Promedio ponderado por fold, luego promedio entre folds
        valores_fold = (
            df[col_train] * w_train +
            df[col_valid] * w_valid +
            df[col_test]  * w_test
        ) / (w_train + w_valid + w_test)

        # Promedio total final entre folds
        resultados[m] = valores_fold.mean()

    return resultados


In [83]:
pesos_folds

{1: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

In [87]:
import pandas as pd

for k in k_folds:
    w = pesos_folds[k]

    # promedio ponderado para ESTE fold (sale como dict)
    res = weighted_avg_metrics_from_df(
        transformers_folds_metrics.loc[[f"transformer_fold_{k}"]],
        w["w_train"],
        w["w_valid"],
        w["w_test"]
    )

    # índice correspondiente en transformers_metrics
    idx = f"transformer_fold_{k}"

    # escribir directamente en el dataset transformers_metrics
    transformers_metrics.loc[idx, cols_base] = [res[m] for m in cols_base]

In [88]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
transformer_fold_1,0.002122,0.00149,0.80553,80.660039,0.836672
transformer_fold_2,0.001956,0.001381,0.830287,76.274642,0.846566
transformer_fold_3,0.001968,0.001378,0.828425,75.351671,0.846498
transformer_fold_4,0.002075,0.001455,0.812202,76.863332,0.842004
transformer_fold_5,0.002294,0.001601,0.776923,80.000594,0.83305


In [89]:
save_metrics(transformers_metrics, "5_3_transformers_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_transformers_metrics.parquet
